In [1]:
import sys
from pathlib import Path

# Setup root progetto
project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pre_tagger import PreTagger
from src.config import (
    TAG_ASSIGN_THRESHOLD,
    TAG_WEIGHT_COSINE,
    TAG_WEIGHT_OVERLAP,
)

print(">> Moduli caricati")

>> Moduli caricati


In [2]:
# Tag originali
candidate_tags = ["Jazz", "Ciclismo", "Astronautica"]

sidecar_path = project_root / "data/processed/armstrong/sidecar_pretag3.json"

BATCH_SIZE = 100

print(f"?> Tag candidati: {candidate_tags}")
print(f"?> Sidecar: {sidecar_path}")

?> Tag candidati: ['Jazz', 'Ciclismo', 'Astronautica']
?> Sidecar: /home/jovyan/tesi_graphrag/data/processed/armstrong/sidecar_pretag3.json


In [3]:
print(f"  >>TagAssignThreshold: {TAG_ASSIGN_THRESHOLD}")
print(f"  >>TagWeightCosine: {TAG_WEIGHT_COSINE}")
print(f"  >>TagWeightOverlap: {TAG_WEIGHT_OVERLAP}")

with PreTagger(sidecar_path=sidecar_path) as pretagger:
    pretagger.run(
        candidate_tags=candidate_tags,
        batch_size=BATCH_SIZE,
    )

  >>TagAssignThreshold: 0.5
  >>TagWeightCosine: 0.75
  >>TagWeightOverlap: 0.25


Pre-tagging 'unilanguage_armstrong':   0%|          | 0/21 [00:00<?, ?chunk/s]

pDB>> tag: Jazz
pDB>> expanded_tag:
  >>>Jazz: Il jazz è un genere musicale nato negli Stati Uniti alla fine del XIX secolo, caratterizzato da un mix di influenze africane, europee e latinoamericane, con strumenti come il saxofono, la tromba, il piano e la batteria, e sotto-categorie come il swing, il bebop e il free jazz, con artisti come Louis Armstrong, Duke Ellington e Miles Davis.
pDB>> tag: Ciclismo
pDB>> expanded_tag:
  >>>Ciclismo: l'attività sportiva che combina velocità, resistenza e tecnica, con sotto-categorie come il mountain bike, il road bike e il BMX, che utilizzano strumenti come le ruote, le maniglie, i freni e le sospensioni, e sono correlate con entità come le gare, le competizioni e le squadre, e termini tecnici come il pedalaggio, la cadenza e la resistenza aerodinamica.
pDB>> tag: Astronautica
pDB>> expanded_tag:
  >>>Astronautica: L'astronautica è un ramo dell'ingegneria aerospaziale che si occupa dello sviluppo, della costruzione e della gestione di veicoli spa

Pre-tagging 'unilanguage_armstrong': 100%|██████████| 21/21 [02:16<00:00,  6.50s/chunk]


In [4]:
from src.sidecar_manager import SidecarManager

verifier = SidecarManager(filepath=str(sidecar_path))
global_tags = verifier.get_global_tags()

# Prova globale
print(f"!>> Tag globali registrati: {len(global_tags)}")
for tag, info in global_tags.items():
    print(f"  - {tag}: conteggio={info.get('count')}, colore={info.get('color')}")

# Campione dei chunk taggati
sidecar_data = verifier.load_data()
overrides = sidecar_data.get("tag_overrides", {})
print(f"\n!>> Totale chunk con tag assegnati: {len(overrides)}")

if overrides:
    print("Sample primi 5 chunk:")
    for chunk_id, info in list(overrides.items())[:5]:
        print(f"  - [{chunk_id}] -> {info.get('user_tags')}")

verifier.release_path()

!>> Tag globali registrati: 3
  - Jazz: conteggio=9, colore=#4e79a7
  - Ciclismo: conteggio=9, colore=#f28e2b
  - Astronautica: conteggio=9, colore=#e15759

!>> Totale chunk con tag assegnati: 19
Sample primi 5 chunk:
  - [louis_armstrong_1] -> ['Jazz']
  - [louis_armstrong_3] -> ['Jazz']
  - [neil_armstrong_5] -> ['Jazz', 'Ciclismo', 'Astronautica']
  - [neil_armstrong_2] -> ['Astronautica']
  - [lance_armstrong_7] -> ['Ciclismo']
